# 06 · Full baseline run

The full baseline answers every item of `run_benchmarks` x `run_splits` at each loop count in `R_grid`, with the same code path as `scripts/mvp.py full`. Items are sharded (`items[i::n]`), one shard per group of `full_gpu_groups` (GPUs 0-7), and each shard writes its own resumable JSONL. This notebook first runs a small sample at every R and checks batched generation against batch 1, then prints the terminal commands of the full run when `RUN_FULL` is True (the run is never started in this kernel), then summarises the result files. The test split stays off until stage S12.

**How to run:** Run All, no edits. It needs the GPUs of `n_copies` (2 copies: GPUs 0-3 all free). The full run is off
(`RUN_FULL = False`); the estimate in step 2 can be skipped with `RUN_ESTIMATE = False`.
Shut down the kernels of other notebooks first: they keep their GPUs. The profiling of step 7 is off too (`RUN_PROFILE = False`).

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# GPU choice must precede CUDA init, which importing adaloop.api already does: the one import outside adaloop.api
from utils.gpus import select_for_notebook

CONFIG = "configs/full_baseline.yaml"   # the config this notebook passes to setup_notebook
# GPUs for the config's n_copies (2: every gpu group, 1: the first free pair); changing it needs a kernel restart
GPU_IDS, N_COPIES = select_for_notebook(CONFIG)

In [ ]:
import os
import socket
import statistics
from dataclasses import asdict

In [ ]:
from adaloop import api

ROOT, cfg = api.setup_notebook("configs/full_baseline.yaml")
print("ROOT", ROOT, "| config_hash", api.config_hash(cfg), "| git", api.git_hash())

In [ ]:
# torch is imported here because the GPU list needs it; the other third-party imports come after this cell
import torch

ON_GPU_HOST = api.is_gpu_host()  # Inara or PACE
print("host", socket.gethostname(), "| on Inara or PACE:", ON_GPU_HOST)
print("chosen GPUs, physical ids CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        total_gib = torch.cuda.get_device_properties(i).total_memory / 1024**3
        print(f"chosen GPU {i} (torch index) | {torch.cuda.get_device_name(i)} | {total_gib:.0f} GiB")
else:
    print("no CUDA GPU visible: the model steps will fail, the rest of the notebook can still be read")
print("gpu_groups", cfg["gpu_groups"], "| only these physical GPUs (0-3 on Inara) may be used")

In [ ]:
import pandas as pd

In [ ]:
# Settings: the full run is off by default because it is long
SAMPLE_N = cfg["sample_items"]  # step 1: dev items per benchmark, run at every R in R_grid
N_CHECK = 8  # step 1: sample items also run at batch 1: half GQA, half TextVQA
R_CHECK = 3  # step 1: loop passes for the batch check
RUN_ESTIMATE = False  # step 2: time a few dev items at every R and estimate the full run
N_ESTIMATE = 16  # step 2: dev items timed per R
RUN_FULL = False  # step 2: True prints the terminal commands of the full run
N_SHARDS = len(cfg["full_gpu_groups"])  # step 2: one terminal and one GPU group per shard
SHARD = (0, N_SHARDS)  # step 2: the shard the estimate is for
N_COST = 16  # step 4: dev items for timing, half GQA, half TextVQA
R_RATIO = (1, 5)  # step 4: prefill time ratio is R_RATIO[1] / R_RATIO[0]
N_REPEATS = 3  # step 4: timed prefill calls per item and R, after one warmup
RATIO_TOL = 0.25  # step 4: allowed gap between measured and analytic ratio (DEC3 §6)
R_THROUGHPUT = 3  # step 4: loop passes for the throughput comparison
THROUGHPUT_BATCHES = (1, cfg["batch_size"])  # step 4
RUN_PROFILE = True  # step 7: True times every stage (about 1.5-2 h on one model copy; the item part is resumable)
PROFILE = cfg["profile"]  # step 7: R, items, methods, k, token scan and trace settings (configs/mvp.yaml)

### Step-1:- Sample run at every R, and the batch check

Before anything long, `SAMPLE_N` dev items per benchmark run at every loop count with all visual tokens (k = 1.0), batched at the
config's `batch_size` as the full run is. The files go to `sample/` under the output folder, apart from the full run's.

In [ ]:
print("[STAGE] load model")
vlms = api.load_models(cfg, N_COPIES)
vlm = vlms[0]

In [ ]:
BATCH = cfg["batch_size"]
sample = {
    "gqa": api.load_gqa(cfg, "dev", limit=SAMPLE_N, root=ROOT),
    "textvqa": api.load_textvqa(cfg, "dev", limit=SAMPLE_N, root=ROOT),
}
sample_dir = ROOT / cfg["out_dir"] / "sample"
assert all(len(items) == SAMPLE_N for items in sample.values()), {b: len(i) for b, i in sample.items()}
{bench: len(items) for bench, items in sample.items()}

In [ ]:
print(f"[STAGE] sample run, R {cfg['R_grid']}, batch {BATCH}")
run_cfg = api.full_run_config(cfg, BATCH)
sample_rows, sample_table = {}, []
for bench, items in sample.items():
    for R in cfg["R_grid"]:
        out = sample_dir / f"sample_{bench}_dev_n{SAMPLE_N}_R{R}_b{BATCH}_{api.config_hash(run_cfg)}.jsonl"
        summary = api.run_evaluation(None, items, run_cfg, R, out, resume=True, batch_size=BATCH, vlms=vlms)
        sample_rows[bench, R] = {row["item_id"]: row for row in api.read_results(out)}
        sample_table.append({"benchmark": bench, "R": R, **{k: summary[k] for k in
                             ("n", "accuracy", "accuracy_lmms", "parse_rate", "forced_rate", "n_think_tokens_mean")}})
sample_table = pd.DataFrame(sample_table)
sample_table

In [ ]:
assert len(sample_table) == len(sample) * len(cfg["R_grid"]) and (sample_table.n == SAMPLE_N).all()
print(f"✓ sample run: {SAMPLE_N} items per benchmark at every R")

#### Batch check: batch 1 against batched

Batched generation pads the prompts to one length. Padding and batched GPU kernels can change the numerics, and a near-tied choice
can flip. `N_CHECK` of the sample items run again at batch 1 at `R_CHECK`, and their answers are compared with the batched rows
above. A mismatch must be reported, not ignored: the full run should not start on batched rows until that is decided.

In [ ]:
print(f"[STAGE] batch 1 on {N_CHECK} sample items, R={R_CHECK}")
check_items = [item for items in sample.values() for item in items[: N_CHECK // 2]]
single_cfg = api.full_run_config(cfg, 1)
out = sample_dir / f"sample_check_dev_n{N_CHECK}_R{R_CHECK}_b1_{api.config_hash(single_cfg)}.jsonl"
api.run_evaluation(None, check_items, single_cfg, R_CHECK, out, resume=True, batch_size=1, vlms=vlms)
single_rows = {row["item_id"]: row for row in api.read_results(out)}

In [ ]:
check_table = pd.DataFrame([
    {
        "benchmark": item.benchmark,
        "item_id": item.item_id,
        "first_answer": item.answers[0],
        "pred_b1": single_rows[item.item_id]["pred"],
        f"pred_b{BATCH}": sample_rows[item.benchmark, R_CHECK][item.item_id]["pred"],
    }
    for item in check_items
])
check_table["equal"] = [a == b for a, b in zip(check_table["pred_b1"], check_table[f"pred_b{BATCH}"])]
check_table

In [ ]:
mismatch = check_table[~check_table["equal"]]
print(f"mismatches: {len(mismatch)} of {len(check_table)}")
mismatch

In [ ]:
assert mismatch.empty, f"batch {BATCH} changed {len(mismatch)} prediction(s); see the table above before the full run"
print(f"✓ batch check: batch 1 and batch {BATCH} give the same predictions on {N_CHECK} items")

### Step-2:- Full run, one terminal per GPU group

The full run is long, so it runs from the CLI, not in this kernel: `scripts/mvp.py full --shard i/n` calls the same `api.run_full` and takes group `i` of `full_gpu_groups` (`configs/full_baseline.yaml`: GPUs 0-1, 2-3, 4-5, 6-7; a big run may use all 8, see `docs/decisions.md`). `RUN_FULL = True` prints the commands, one per terminal on Inara.

Before starting them, shut down this kernel and other notebook kernels: they hold GPUs 0-3. Each shard writes its own files and is resumable: rerun the same command after an interruption. Items finished under another shard count (an earlier 2-shard run) are skipped too; stop those processes before changing the count.

#### Estimate first

`estimate_full_run` times `N_ESTIMATE` dev items at every R in `R_grid` with the run's batch size, and multiplies by the items of
shard `SHARD` that have no result row yet. It writes no result files. Read the estimate before starting the run: the model may
think up to `think_budget` tokens per item, so a run is long.

In [ ]:
if RUN_ESTIMATE:
    print(f"[STAGE] estimate, shard {SHARD[0]}/{SHARD[1]}")
    estimate = api.estimate_full_run(vlm, cfg, ROOT, shard=SHARD, n_items=N_ESTIMATE)
pd.DataFrame(estimate["per_R"]) if RUN_ESTIMATE else "skipped: RUN_ESTIMATE is False"

In [ ]:
if RUN_FULL:
    print(f"run each line in its own terminal ({N_SHARDS} shards, GPU groups {cfg['full_gpu_groups']}):")
    for i in range(N_SHARDS):
        print(f"cd {ROOT} && python scripts/mvp.py full --shard {i}/{N_SHARDS}")
else:
    print("skipped: RUN_FULL is False")

### Step-3:- Summary per benchmark x split x R

Accuracy (headline and lmms-eval), parse rate, forced-answer rate (`forced_rate`, %), mean thinking tokens
(`n_think_tokens_mean`) and seconds per item, read from every shard file of this run config (`batch_size` from the config).
dev and ctrl_train are kept apart, and each item counts once. For batched rows, s/item is batch time divided by batch size, measured on
2x A40 with balanced placement. The table is empty until result files exist.

In [ ]:
summary = api.summarize_full(cfg, ROOT)
full_table = (
    pd.DataFrame(summary)
    if summary
    else f"no result files yet for batch_size {cfg['batch_size']} in {cfg['out_dir']}"
)
full_table  # forced_rate and n_think_tokens_mean show how often and how long the thinking ran

### Step-4:- Cost model (stage S6)

The controller is compared at matched FLOPs, so we need the cost of every (tokens, R) choice. FLOPs are computed from the
loaded model's parameter shapes (`docs/DEC3_MVP.md` §6) and then checked against measured time. All times in this step are
measured on 2x A40 with balanced placement; the GPUs actually in use are printed below.

In [ ]:
print("[STAGE] load model (reused if step 1 loaded it)")
vlms = api.load_models(cfg, N_COPIES)
vlm = vlms[0]
GPUS = sorted(set(vlm.model.hf_device_map.values()))
GPU_NOTE = f"{len(GPUS)}x {' / '.join(sorted({torch.cuda.get_device_name(i) for i in GPUS}))}, {cfg['device_map']} placement"
print("GPUs in use (torch ids):", GPUS, "|", GPU_NOTE)

#### Dev items and inputs

One set of dev items serves the whole step. GQA and TextVQA alternate, so every batch mixes both. The inputs are built once,
with the same code as the evaluation runner.

In [ ]:
gqa_items = api.load_gqa(cfg, "dev", limit=N_COST // 2, root=ROOT)
textvqa_items = api.load_textvqa(cfg, "dev", limit=N_COST // 2, root=ROOT)
cost_items = [item for pair in zip(gqa_items, textvqa_items) for item in pair]
cost_inputs = [api.item_inputs(vlm, item, cfg) for item in cost_items]
prompt_lens = [inputs["input_ids"].shape[1] for inputs in cost_inputs]  # T per item, all visual tokens
assert len(cost_inputs) == N_COST, len(cost_inputs)
print(f"{N_COST} items | prompt tokens: min {min(prompt_lens)}, median {statistics.median(prompt_lens)}, max {max(prompt_lens)}")

#### Active parameters per part

Parameters used per token (multiply-adds). MoE experts count only the share that is active (8 of 128). `core` is one loop
pass; `core_message` is what every pass after the first adds; `embed` is a lookup and costs no FLOPs.

In [ ]:
params = api.cost_params(vlm.model)
PARTS = ["embed", "prelude", "core", "core_message", "coda", "lm_head", "vit_patch", "vit_layers", "vit_projector"]
pd.Series({part: asdict(params)[part] / 1e6 for part in PARTS}, name="million parameters").round(2)

#### FLOPs per R for a typical item

The typical item is the one with the median prompt length. Its output length (thinking included) is taken from one real answer at
R = 3, and held fixed over R. If that answer was forced (the thinking budget ran out), the vision tower and the prompt forward ran
a second time; this shows up as `flops_reprefill`. The vision tower does not depend on R or on pruning, so its share falls as R
grows.

In [ ]:
typical = sorted(range(N_COST), key=lambda i: prompt_lens[i])[N_COST // 2]
typical_inputs, n_tokens = cost_inputs[typical], prompt_lens[typical]
grid_thw = typical_inputs["image_grid_thw"]
api.set_loops(vlm, api.REF_R)
gen = api.generate_answers(vlm, [typical_inputs], cfg)[0]
n_out, n_reprefill = len(gen.token_ids), gen.n_reprefill_tokens  # n_reprefill is 0 unless the answer was forced
print(f"item {cost_items[typical].item_id} | prompt tokens {n_tokens} | output tokens {n_out} | forced {gen.forced} (reprefill {n_reprefill}) | image grid {grid_thw.tolist()}")

In [ ]:
flops_by_R = pd.DataFrame(
    [api.flops_total(params, n_tokens, n_out, R, grid_thw, n_reprefill=n_reprefill) for R in cfg["R_grid"]],
    index=pd.Index(cfg["R_grid"], name="R"),
)
flops_table = (flops_by_R / 1e9).add_suffix(" (G)")
flops_table["vit_share"] = flops_by_R["flops_vit"] / flops_by_R["flops_total"]
flops_table.round(3)

In [ ]:
assert flops_by_R["flops_total"].is_monotonic_increasing and flops_by_R["flops_vit"].nunique() == 1
# cost_ref is one pass over the item; a forced answer adds a second vision and prompt forward on top
extra = 0
if n_reprefill:
    extra = api.flops_vit(params, grid_thw) + api.flops_prefill(params, n_reprefill, api.REF_R)
    assert flops_by_R.loc[api.REF_R, "flops_reprefill"] == extra
assert flops_by_R.loc[api.REF_R, "flops_total"] == api.cost_ref(params, n_tokens, n_out, grid_thw) + extra
print("✓ cost model: FLOPs grow with R, ViT cost is fixed, R = 3 equals cost_ref (plus the reprefill if forced)")

#### Measured vs analytic prefill time ratio

For each item the prompt forward is timed (time to first token) at both loop counts. All visual tokens are kept through
`prune_inputs`, so the vision tower runs outside the timer. One warmup call, then the median of the repeats. Keep the GPUs
otherwise idle. The test of stage S6: the median measured ratio is within ±25% of the analytic one.

In [ ]:
R_LOW, R_HIGH = R_RATIO
print(f"[STAGE] prefill timing at R={R_LOW} and R={R_HIGH} on {N_COST} items | measured on {GPU_NOTE}")
ratio_rows = []
for i, (item, inputs, n) in enumerate(zip(cost_items, cost_inputs, prompt_lens)):
    low_s = api.time_prefill(vlm, inputs, R_LOW, N_REPEATS)
    high_s = api.time_prefill(vlm, inputs, R_HIGH, N_REPEATS)
    analytic = api.flops_prefill(params, n, R_HIGH) / api.flops_prefill(params, n, R_LOW)
    ratio_rows.append({"item_id": item.item_id, "n_tokens": n, "low_s": low_s, "high_s": high_s,
                       "measured": high_s / low_s, "analytic": analytic})
    print(f"[{i + 1}/{N_COST}] T={n} | R={R_LOW}: {low_s:.3f}s | R={R_HIGH}: {high_s:.3f}s | ratio {high_s / low_s:.2f}")
api.set_loops(vlm, api.REF_R)
ratios = pd.DataFrame(ratio_rows)
ratios

In [ ]:
measured, analytic = ratios["measured"].median(), ratios["analytic"].median()
print(f"median R{R_HIGH}/R{R_LOW} prefill ratio | measured {measured:.2f} | analytic {analytic:.2f} | measured on {GPU_NOTE}")
assert abs(measured / analytic - 1) <= RATIO_TOL, f"measured {measured:.2f} vs analytic {analytic:.2f}"
print(f"✓ test_flops: measured prefill time ratio is within ±{RATIO_TOL:.0%} of the analytic ratio")

#### Throughput: batch 1 vs batch 8

Seconds per item for the same preloaded inputs at one loop count, generated as a run does (`generate_answers`, thinking budget
included) at batch 1 and at the config's `batch_size` (8). Wall clock, including the vision tower and decoding, excluding data
loading. One warmup call per batch size. A batch runs until its longest answer ends. Measured on 2x A40 with balanced placement;
these seconds set the item counts of the grid.

In [ ]:
print(f"[STAGE] throughput at R={R_THROUGHPUT}, batch sizes {THROUGHPUT_BATCHES} | measured on {GPU_NOTE}")
throughput = pd.DataFrame([
    api.time_throughput(vlm, cost_inputs, R_THROUGHPUT, batch_size, cfg)
    for batch_size in THROUGHPUT_BATCHES
])
throughput["gpus"] = GPU_NOTE
throughput["config_hash"] = api.config_hash(cfg)
throughput["git_hash"] = api.git_hash()
api.set_loops(vlm, api.REF_R)
throughput

In [ ]:
throughput_path = ROOT / cfg["out_dir"] / f"throughput_R{R_THROUGHPUT}_n{N_COST}_{api.config_hash(cfg)}.csv"
throughput_path.parent.mkdir(parents=True, exist_ok=True)
throughput.to_csv(throughput_path, index=False)
assert throughput_path.is_file() and (throughput["s_per_item"] > 0).all()
for row in throughput.itertuples():
    print(f"batch {row.batch_size}: {row.s_per_item:.2f} s/item | measured on {row.gpus}")
print(f"✓ throughput saved to {throughput_path.relative_to(ROOT)}")

#### Cost of every grid cell

The grid (S7) crosses the kept fraction `k` with the loop passes `R`, under the compressor frozen in S5 (divprune; before the freeze this used
`text_mean`). FLOPs per cell: vision tower (full image for methods that prune after it, smaller for `lowres`), prompt
forward and decode at the packed length, and the compression step. The grid settings come from `configs/mvp.yaml`.

In [ ]:
grid_cfg = api.load_config(ROOT / "configs/mvp.yaml")
if grid_cfg["compressor"] == "none":
    grid_cfg["compressor"] = "text_mean"  # fallback only: mvp.yaml is frozen to divprune (S5), so this branch is not taken
print("compressor", grid_cfg["compressor"], "| k", grid_cfg["k_grid"], "| R", grid_cfg["R_grid"])
cells_cost = pd.DataFrame(api.cost_grid(vlm, cost_items[typical], grid_cfg, n_out))
shown = cells_cost[["k", "R", "n_vis", "n_input"]].copy()
for part in ["flops_vit", "flops_prefill", "flops_decode", "flops_compression", "flops_total"]:
    shown[part.replace("flops_", "") + " (G)"] = (cells_cost[part] / 1e9).round(3)
shown

In [ ]:
total = cells_cost.set_index(["k", "R"]).flops_total
for k in grid_cfg["k_grid"]:
    assert total.loc[k].is_monotonic_increasing, f"k={k}: FLOPs must grow with R"
for R in grid_cfg["R_grid"]:
    assert total.xs(R, level="R").sort_index().is_monotonic_increasing, f"R={R}: FLOPs must grow with k"
if grid_cfg["compressor"] != "lowres":
    assert cells_cost.flops_vit.nunique() == 1, "the vision tower sees the full image in every cell"
print("✓ grid cost: grows with k and with R")

#### Time per item over all cells, and the grid size

Each timed item runs every cell exactly as the grid does (features on), on one model copy. `RUN_GRID_TIME` is off: first
run it with `N_GRID_TIME = 4` as a sample, then 50 (the S6 done-when). The estimate covers ctrl_train + dev plus the
noise re-run over `N_COPIES` copies; it assumes the copies do not slow each other, so it is a lower bound.

The grid runs batched, so its rows have no per-item times. The same cell then times `latency_items` items per benchmark
at batch 1 on one copy; the medians per (k, R) are the grid's timing columns, saved to `grid_timing/grid_timing.csv`
under the run's output folder (features on, as in the grid).

In [ ]:
RUN_GRID_TIME = False
N_GRID_TIME = 4  # 4 as a sample, then 50
if RUN_GRID_TIME:
    print(f"[STAGE] grid timing, {N_GRID_TIME} items | measured on {GPU_NOTE}")
    time_items = api.load_gqa(cfg, "dev", limit=N_GRID_TIME // 2, root=ROOT) + \
        api.load_textvqa(cfg, "dev", limit=N_GRID_TIME // 2, root=ROOT)
    timed_rows, s_per_item = api.time_grid(vlm, time_items, grid_cfg)
    print(f"[STAGE] timing columns: {grid_cfg['latency_items']} items per benchmark, batch 1, one copy")
    latency_set = api.load_gqa(cfg, "dev", limit=grid_cfg["latency_items"], root=ROOT) + \
        api.load_textvqa(cfg, "dev", limit=grid_cfg["latency_items"], root=ROOT)
    latency_rows, _ = api.time_grid(vlm, latency_set, grid_cfg, batch_size=1)
    api.set_loops(vlm, api.REF_R)
else:
    print("skipped: RUN_GRID_TIME is False")

In [ ]:
if RUN_GRID_TIME:
    n_grid_items = sum(grid_cfg["splits"]["sizes"][s] for s in ["ctrl_train", "dev"]) * len(grid_cfg["benchmarks"])
    hours = api.estimate_grid_hours(s_per_item, n_grid_items, grid_cfg["n_rerun"], N_COPIES)
    assert len(timed_rows) == N_GRID_TIME * len(grid_cfg["k_grid"]) * len(grid_cfg["R_grid"])
    print(f"## <date> · S6 · budget (git {api.git_hash()}, {GPU_NOTE})")
    print(f"- {s_per_item:.1f} s per item for all cells ({N_GRID_TIME} dev items, compressor {grid_cfg['compressor']})")
    print(f"- grid: {n_grid_items} items + {grid_cfg['n_rerun']} re-run, {N_COPIES} copies: at least {hours:.1f} h")
    print("✓ grid timing: every cell timed")
    timing = pd.DataFrame(api.latency_table(latency_rows, ("selector", "k", "R")))
    assert len(timing) == len(grid_cfg["k_grid"]) * len(grid_cfg["R_grid"]) and (timing.latency_n == len(latency_set)).all()
    timing_path = ROOT / cfg["out_dir"] / "grid_timing" / "grid_timing.csv"
    timing_path.parent.mkdir(parents=True, exist_ok=True)
    timing.to_csv(timing_path, index=False)
    print(f"✓ grid timing columns: {len(timing)} cells -> {timing_path.relative_to(ROOT)}")
timing.round(3) if RUN_GRID_TIME else "skipped"

### Step-5:- Figure F04

Cost model: FLOPs per loop count R for one typical item (median prompt length), and the analytic against measured prefill time ratio from step 4. The heatmap covers every grid cell (k x R).

In [ ]:
api.set_paper_style()
fig = api.plot_cost_model(cells_cost, ratios, R_RATIO, RATIO_TOL)

In [ ]:
pdf_path, png_path = api.save_figure(fig, "F04_cost_model", final=False)

In [ ]:
assert pdf_path.is_file() and png_path.is_file(), f"missing {pdf_path} or {png_path}"
print("✓ F04_cost_model draft saved as PDF and PNG (final=False, paper/ untouched)")

### Step-6:- Figure F05

Accuracy against loop passes R with all visual tokens, one panel per benchmark (the loop-depth sweep). Check `n` per row before using the figure: a partial run looks like a full one, so every row must have the full item count.

In [ ]:
pd.DataFrame(summary)[["benchmark", "split", "R", "n", "accuracy"]] if summary else "skipped: no result files yet (run step 2 first)"

In [ ]:
if summary:
    api.set_paper_style()
    fig = api.plot_accuracy_vs_r(pd.DataFrame(summary))
else:
    print("skipped: summary is empty, no result files yet")

In [ ]:
if summary:
    pdf_path, png_path = api.save_figure(fig, "F05_accuracy_vs_r", final=False)
else:
    print("skipped: F05 not saved, summary is empty")

In [ ]:
if summary:
    assert pdf_path.is_file() and png_path.is_file(), f"missing {pdf_path} or {png_path}"
    print("✓ F05_accuracy_vs_r draft saved as PDF and PNG (final=False, paper/ untouched)")
else:
    print("skipped: F05 not saved, summary is empty")

### Step-7:- Profiling: where the time goes

For `none` and each compression method at several k, every stage of one answer is timed: vision tower, projector, token selection, building the pruned inputs, prefill (prelude, each core pass, coda) and decoding. Each answer runs twice through the same production code: plain (wall time) and with forward hooks (stages). Timing only: no accuracy, and the rows go to their own `profile/` folder. `cls_iqr` is left out: it has no keep fraction `k`.

One model copy (copy 0) is used. Keep its GPUs, and the other copy's, idle. `RUN_PROFILE` is set in the settings cell.

In [ ]:
PROFILE_DIR = ROOT / cfg["out_dir"] / "profile"
if RUN_PROFILE:
    print("[STAGE] load model (reused) and dev items")
    vlm = api.load_models(cfg, N_COPIES)[0]
    profile_set = {
        "gqa": api.load_gqa(cfg, "dev", limit=PROFILE["n_items"], root=ROOT),
        "textvqa": api.load_textvqa(cfg, "dev", limit=PROFILE["n_items"], root=ROOT),
    }
{bench: len(items) for bench, items in profile_set.items()} if RUN_PROFILE else "skipped: RUN_PROFILE is False"

#### One item first

`none` and the trace method at k = 0.25. The columns are milliseconds. Stages inside the model are GPU times between hooks; `select_ms` and `build_ms` are synchronized wall time (CPU work included). `handoff_ms` (the move between the two GPUs) and `select_spacy_ms` / `select_attention_ms` are parts of other columns. `overhead_ms` is the plain wall time minus the stages.

In [ ]:
if RUN_PROFILE:
    print("[STAGE] profile one item")
    one = profile_set["gqa"][0]
    n_hooks = api.count_hooks(vlm)
    one_cells = [("none", 1.0), (PROFILE["trace"]["method"], 0.25)]
    api.profile_item(vlm, one, cfg, "none", 1.0, PROFILE["R"])  # warmup
    one_rows = [api.profile_item(vlm, one, cfg, method, k, PROFILE["R"]) for method, k in one_cells]
pd.DataFrame(one_rows).T if RUN_PROFILE else "skipped"

In [ ]:
if RUN_PROFILE:
    for row in one_rows:
        passes = [key for key in row if key.startswith("core")]
        assert row["same_output"], f"{row['method']}: the hooked run answered differently"
        assert len(passes) == PROFILE["R"] and all(row[key] > 0 for key in passes), passes
        assert row["stage_sum_ms"] <= row["wall_profiled_ms"], (row["stage_sum_ms"], row["wall_profiled_ms"])
    assert api.count_hooks(vlm) == n_hooks, "profiling left hooks on the model"
    print(f"✓ profile of one item: {PROFILE['R']} core passes, stages within the wall time, same output, hooks removed")
else:
    print("skipped")

#### All items and cells

Medians over the items of both benchmarks. `pct_*` is the median stage over the median wall time (medians of parts need not add to 100). `compression_overhead_ms` is what compression adds (selection and input building); `saved_ms` is the wall time of `none` minus the cell's, per item, so it already pays that overhead. A negative value means compression made the answer slower. This part is resumable; the token scan and the trace are not.

In [ ]:
if RUN_PROFILE:
    print(f"[STAGE] profile {PROFILE['n_items']} items per benchmark x {len(api.profile_cells(cfg))} cells")
    profile_rows = api.profile_items(vlm, profile_set, cfg, PROFILE_DIR)
    api.set_loops(vlm, api.REF_R)
    stage_table = pd.DataFrame(api.profile_table(profile_rows)).set_index(["method", "k"])
    MS = [c for c in stage_table.columns if c.endswith(("_ms", "_per_token")) and not c.startswith("pct_") and c not in ("saved_ms", "compression_overhead_ms")]
stage_table[MS].round(1) if RUN_PROFILE else "skipped"

In [ ]:
stage_table[[c for c in stage_table.columns if c.startswith("pct_")]].round(1) if RUN_PROFILE else "skipped"

In [ ]:
SUMMARY = ["n", "n_vis", "n_new", "wall_ms", "compression_overhead_ms", "saved_ms", "saved_pct", "peak_mem_gb", "kv_cache_bytes"]
stage_table[SUMMARY].round(2) if RUN_PROFILE else "skipped"

In [ ]:
if RUN_PROFILE:
    assert len(stage_table) == len(api.profile_cells(cfg)) and (stage_table["n"] == 2 * PROFILE["n_items"]).all()
    changed = [(r["item_id"], r["method"], r["k"]) for r in profile_rows if not r["same_output"]]
    assert not changed, f"hooked and plain runs answered differently: {changed[:5]}"
    api.save_table(stage_table.reset_index(), "profile_stages")
    print(f"✓ profile: {len(profile_rows)} rows, every cell has every item, hooks never changed an answer")
else:
    print("skipped")

#### Token-count scan: if we cut N tokens, is the decoder faster?

All visual tokens (`none`) against `random` at exact kept counts, per R and batch size, with the same decode length for every answer. The vision tower runs outside the timed part. 0 kept tokens is the prompt with the image placeholders removed (the model's text-only path; `compress` itself keeps at least one). An item with fewer tokens than asked keeps them all, which `n_vis` shows. A decode step produces one token per row of the batch.

In [ ]:
if RUN_PROFILE:
    n_scan = PROFILE["scan"]["n_items"]
    scan_items = [item for pair in zip(profile_set["gqa"][:n_scan], profile_set["textvqa"][:n_scan]) for item in pair]
    print(f"[STAGE] token scan on {len(scan_items)} items")
    scan_rows = api.run_token_scan(vlm, scan_items, cfg, PROFILE_DIR)
    api.set_loops(vlm, api.REF_R)
    scan_table = pd.DataFrame(api.token_scan_table(scan_rows))
scan_table.round(1) if RUN_PROFILE else "skipped"

In [ ]:
if RUN_PROFILE:
    scan = PROFILE["scan"]
    assert len(scan_table) == (1 + len(scan["kept"])) * len(scan["R_list"]) * len(scan["batch_sizes"])
    assert all(r["n_new"] == scan["n_decode"] for r in scan_rows), "some answers stopped early"
    random_rows = scan_table[(scan_table["method"] == "random") & (scan_table["batch_size"] == 1)]
    assert (random_rows["n_vis"] <= random_rows["kept"]).all()
    api.save_table(scan_table, "profile_token_scan")
    print(f"✓ token scan: {len(scan_table)} settings, {scan['n_decode']} tokens decoded in each")
else:
    print("skipped")

#### Figure F04b

In [ ]:
if RUN_PROFILE:
    api.set_paper_style()
    fig = api.plot_token_scan(scan_table)
    scan_pdf, scan_png = api.save_figure(fig, "F04b_token_scan", final=False)
    assert scan_pdf.is_file() and scan_png.is_file()
    print("✓ F04b_token_scan draft saved")
else:
    print("skipped")

#### torch.profiler trace of one item

The same item at k = 1.0 and k = 0.25. The Chrome trace files open in `chrome://tracing` or Perfetto. `busy_ms` is the time any GPU ran a kernel or a copy; `idle_ms` is the rest of the wall time, profiler overhead included. The second table is the 15 kernels with the most time at each k.

In [ ]:
if RUN_PROFILE:
    TRACE = PROFILE["trace"]
    print(f"[STAGE] torch.profiler trace, {TRACE['method']} at k = {TRACE['ks']}")
    trace_summary, trace_kernels = api.trace_item(vlm, one, cfg, TRACE["method"], TRACE["ks"], PROFILE["R"], PROFILE_DIR)
    api.set_loops(vlm, api.REF_R)
pd.DataFrame(trace_summary) if RUN_PROFILE else "skipped"

In [ ]:
if RUN_PROFILE:
    for k, group in pd.DataFrame(trace_kernels).groupby("k", sort=False):
        print(f"\nk = {k}: top {len(group)}")
        print(group.drop(columns="k").to_string(index=False, float_format="%.2f"))
    assert len(trace_summary) == len(TRACE["ks"]) and all(os.path.isfile(s["trace"]) for s in trace_summary)
    assert all(kernel["kind"] == "gpu_kernel" for kernel in trace_kernels), "no GPU kernel in the trace"
    print("✓ trace files and kernel tables written")
else:
    print("skipped")